# 014 · Load new landing files into bronze with Auto Loader
Reads new JSON files from `/Volumes/<catalog>/bronze/landing/complaints/` and appends them to
`<catalog>.bronze.complaints_bronze`. Runs after `011` in the daily job.

### How it works
- **Only new files:** `includeExistingFiles = false`, so the 365-day backfill already loaded by `012`
  (COPY INTO) isn't loaded again. The checkpoint records every file loaded, so re-runs never duplicate.
- **Schema:** column types are inferred (`inferColumnTypes`) and saved in the schema location.
  New columns are added automatically (`addNewColumns`); the run fails once when one appears and succeeds on retry.
  Values that don't fit their column go to `_rescued_data` instead of being lost.
- **Lineage columns:** `source_file` (which file a row came from) and `loaded_at` (when it was loaded).
- **Trigger:** `availableNow`. It processes everything new, then stops, which fits a once-a-day job.

### Parameters & paths
`catalog` widget (default `cfpb_dev`), filled in by the job.
Checkpoint: `/Volumes/<catalog>/bronze/landing/_checkpoints/complaints_bronze`

**Don't re-run `012` after this is live.** COPY INTO only tracks its own loads, so it would load
Auto Loader's files a second time.


In [0]:
from pyspark.sql.functions import col, current_timestamp

# Input box for the catalog; the job fills it in, and the default is for manual runs
dbutils.widgets.text("catalog", "cfpb_dev")
catalog = dbutils.widgets.get("catalog")

LANDING    = f"/Volumes/{catalog}/bronze/landing/complaints"                    # folder to watch
CHECKPOINT = f"/Volumes/{catalog}/bronze/landing/_checkpoints/complaints_bronze" # Auto Loader's memory

df = (spark.readStream.format("cloudFiles")                         # cloudFiles = Auto Loader
      .option("cloudFiles.format", "json")                           # files are JSON
      .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")  # where it saves the column layout
      .option("cloudFiles.inferColumnTypes", "true")                 # guess numbers/dates, not all text
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")     # new column? add it
      .option("cloudFiles.includeExistingFiles", "false")            # skip the 365 already loaded
      .load(LANDING)
      .select("*",
              col("_metadata.file_name").alias("source_file"),       # which file each row came from
              current_timestamp().alias("loaded_at")))               # when it was loaded

(df.writeStream
   .option("checkpointLocation", CHECKPOINT)   # remember which files are done
   .option("mergeSchema", "true")              # allow new columns to be added to the table
   .trigger(availableNow=True)                 # load everything new, then stop
   .toTable(f"{catalog}.bronze.complaints_bronze"))